# 02 数据清洗与存储

本 Notebook 完成以下任务：
1. 单表清洗（缺失值、日期格式、数据类型、重复值、离群值）
2. 宽表与长表转换
3. 多表合并
4. CSV存储（方式A）
5. SQLite存储（方式C，进阶）

In [1]:
import pandas as pd
import numpy as np
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# 股票信息字典
stock_info = {
    '601398': ('工商银行', '银行'),
    '000001': ('平安银行', '银行'),
    '000625': ('长安汽车', '汽车'),
    '600104': ('上汽集团', '汽车'),
    '000002': ('万科A',   '房地产'),
    '600048': ('保利发展', '房地产'),
    '000858': ('五粮液',   '白酒'),
    '600519': ('贵州茅台', '白酒'),
    '601857': ('中国石油', '能源'),
    '600028': ('中国石化', '能源'),
}

codes = list(stock_info.keys())
print(f'共 {len(codes)} 只股票待清洗')

共 10 只股票待清洗


---
## 3.1 单表清洗

对每只股票的原始数据进行清洗，包括：缺失值检测与处理、日期格式统一、数据类型检查、重复值处理、离群值标注。

### 3.1.1 缺失值检测

In [2]:
# 读取所有股票数据并检测缺失值
stock_dfs = {}
missing_report = []

for code in codes:
    filepath = f'data/stock/stock_{code}.csv'
    if not os.path.exists(filepath):
        print(f'警告: {filepath} 不存在，跳过')
        continue
    df = pd.read_csv(filepath)
    stock_dfs[code] = df
    
    # 统计缺失值
    for col in df.columns:
        n_missing = df[col].isna().sum()
        # baostock返回空字符串而非NaN，也需检测
        if df[col].dtype == object:
            n_empty = (df[col] == '').sum()
            n_missing += n_empty
        pct_missing = n_missing / len(df) * 100
        missing_report.append({
            'code': code,
            'name': stock_info[code][0],
            'column': col,
            'missing_count': n_missing,
            'missing_pct': f'{pct_missing:.2f}%'
        })

missing_df = pd.DataFrame(missing_report)
print('=== 缺失值检测报告 ===')
missing_nonzero = missing_df[missing_df['missing_count'] > 0]
if len(missing_nonzero) > 0:
    print(missing_nonzero.to_string(index=False))
else:
    print('所有股票数据均无缺失值！')

print('\n缺失值可能原因分析：')
print('- 停牌：A股市场停牌期间不会有交易数据，baostock数据已排除停牌日')
print('- 节假日：非交易日无数据，这是正常现象，不属于缺失')
print('- 数据源问题：baostock返回空字符串表示缺失，需转换为NaN')

=== 缺失值检测报告 ===
所有股票数据均无缺失值！

缺失值可能原因分析：
- 停牌：A股市场停牌期间不会有交易数据，baostock数据已排除停牌日
- 节假日：非交易日无数据，这是正常现象，不属于缺失
- 数据源问题：baostock返回空字符串表示缺失，需转换为NaN


### 3.1.2 缺失值处理

In [3]:
# 缺失值处理：
# 1. baostock返回的空字符串替换为NaN
# 2. 向前填充（ffill），因为停牌期间股票价格实质不变
# 3. 删除仍为NaN的行（序列开头可能无法填充）

for code in stock_dfs:
    df = stock_dfs[code].copy()
    # 将空字符串替换为NaN
    df = df.replace('', np.nan)
    n_before = df.isna().sum().sum()
    if n_before > 0:
        df = df.ffill()
        df = df.dropna()
        n_after = df.isna().sum().sum()
        print(f'{stock_info[code][0]}({code}): 缺失值 {n_before} -> {n_after}')
    stock_dfs[code] = df

print('\n缺失值处理完成！采用向前填充策略，因为停牌期间股票价格实质不变，ffill符合经济含义。')


缺失值处理完成！采用向前填充策略，因为停牌期间股票价格实质不变，ffill符合经济含义。


### 3.1.3 日期格式统一

In [4]:
# 将日期列统一为 datetime64 格式，并设为索引
for code in stock_dfs:
    df = stock_dfs[code].copy()
    dtype_before = df['日期'].dtype
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期')
    stock_dfs[code] = df
    print(f'{stock_info[code][0]}({code}): 日期类型 {dtype_before} -> datetime64, 索引范围 {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')

print('\n日期格式统一完成！')

工商银行(601398): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
平安银行(000001): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
长安汽车(000625): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
上汽集团(600104): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
万科A(000002): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
保利发展(600048): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
五粮液(000858): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
贵州茅台(600519): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
中国石油(601857): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21
中国石化(600028): 日期类型 str -> datetime64, 索引范围 2020-01-02 ~ 2026-05-21

日期格式统一完成！


### 3.1.4 数据类型检查

In [5]:
# 检查价格、成交量列是否为数值型
numeric_cols = ['开盘价', '收盘价', '最高价', '最低价', '成交量', '成交额']
type_report = []

for code in stock_dfs:
    df = stock_dfs[code]
    for col in numeric_cols:
        dtype = df[col].dtype
        is_numeric = pd.api.types.is_numeric_dtype(dtype)
        type_report.append({
            'code': code,
            'name': stock_info[code][0],
            'column': col,
            'dtype': str(dtype),
            'is_numeric': is_numeric
        })
        if not is_numeric:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            stock_dfs[code] = df

type_df = pd.DataFrame(type_report)
non_numeric = type_df[~type_df['is_numeric']]
if len(non_numeric) > 0:
    print('以下列非数值型，已转换：')
    print(non_numeric.to_string(index=False))
else:
    print('所有价格和成交量列均为数值型，无需转换。')

print('\n转换后数据类型检查：')
print(stock_dfs[codes[0]].dtypes)

所有价格和成交量列均为数值型，无需转换。

转换后数据类型检查：
开盘价    float64
收盘价    float64
最高价    float64
最低价    float64
成交量      int64
成交额    float64
dtype: object


### 3.1.5 重复值处理

In [6]:
# 检测并删除重复行
for code in stock_dfs:
    df = stock_dfs[code]
    n_dup = df.index.duplicated().sum()
    if n_dup > 0:
        print(f'{stock_info[code][0]}({code}): 发现 {n_dup} 个重复日期，已删除')
        df = df[~df.index.duplicated(keep='first')]
        stock_dfs[code] = df
    else:
        print(f'{stock_info[code][0]}({code}): 无重复行')

print('\n重复值处理完成！')

工商银行(601398): 无重复行
平安银行(000001): 无重复行
长安汽车(000625): 无重复行
上汽集团(600104): 无重复行
万科A(000002): 无重复行
保利发展(600048): 无重复行
五粮液(000858): 无重复行
贵州茅台(600519): 无重复行
中国石油(601857): 无重复行
中国石化(600028): 无重复行

重复值处理完成！


### 3.1.6 离群值标注

In [7]:
# 计算日收益率，标注涨跌幅超过±20%的记录
extreme_report = []

for code in stock_dfs:
    df = stock_dfs[code].copy()
    df['log_return'] = np.log(df['收盘价'] / df['收盘价'].shift(1))
    df['is_extreme'] = df['log_return'].abs() > 0.20
    n_extreme = df['is_extreme'].sum()
    stock_dfs[code] = df
    
    if n_extreme > 0:
        extreme_dates = df[df['is_extreme']].index.tolist()
        extreme_returns = df[df['is_extreme']]['log_return'].tolist()
        for d, r in zip(extreme_dates, extreme_returns):
            extreme_report.append({
                'code': code,
                'name': stock_info[code][0],
                'date': str(d.date()) if hasattr(d, 'date') else str(d)[:10],
                'log_return': f'{r:.4f}',
                'pct_change': f'{r*100:.2f}%'
            })
        print(f'{stock_info[code][0]}({code}): {n_extreme} 个极端收益率')
    else:
        print(f'{stock_info[code][0]}({code}): 无极端收益率')

if extreme_report:
    extreme_df = pd.DataFrame(extreme_report)
    print('\n=== 极端收益率详情 ===')
    print(extreme_df.to_string(index=False))
    print('\n可能成因分析：')
    print('- 后复权价格调整：分红送股除权日，后复权价可能产生较大跳变')
    print('- ST股涨跌停：特殊情况下的涨跌幅限制可能不同')
    print('- 重大事件：如重组、退市风险等导致连续涨跌停')
else:
    print('\n所有股票均无单日涨跌幅超过±20%的极端情况。')

print('\n离群值标注完成！is_extreme列标记为True的记录不删除，但需在后续分析中注意。')

工商银行(601398): 无极端收益率
平安银行(000001): 无极端收益率
长安汽车(000625): 无极端收益率
上汽集团(600104): 无极端收益率
万科A(000002): 无极端收益率
保利发展(600048): 无极端收益率
五粮液(000858): 无极端收益率
贵州茅台(600519): 无极端收益率
中国石油(601857): 无极端收益率
中国石化(600028): 无极端收益率

所有股票均无单日涨跌幅超过±20%的极端情况。

离群值标注完成！is_extreme列标记为True的记录不删除，但需在后续分析中注意。


---
## 3.2 宽表与长表转换

In [8]:
# 将10只股票的收盘价合并为宽表（日期为索引，每列一只股票）
close_dict = {}
for code in stock_dfs:
    close_dict[f'{code}_{stock_info[code][0]}'] = stock_dfs[code]['收盘价']

wide_df = pd.DataFrame(close_dict)
wide_df.index.name = 'date'

print('=== 宽表（Wide Format）===')
print(f'形状: {wide_df.shape}')
print(wide_df.head())
print(f'\n宽表适合的操作：')
print('- 计算多只股票的相关系数矩阵')
print('- 同时绘制多只股票的走势对比图')
print('- 构建投资组合权重计算')
print('- 横截面分析（同一时间点比较不同股票）')

=== 宽表（Wide Format）===
形状: (1544, 10)
            601398_工商银行  000001_平安银行  000625_长安汽车  600104_上汽集团  000002_万科A  \
date                                                                         
2020-01-02     4.151460    13.684725     5.424149    20.212888   26.594324   
2020-01-03     4.165368    13.936193     5.268456    20.154348   26.177767   
2020-01-06     4.151460    13.846962     5.238321    19.535501   25.736706   
2020-01-07     4.179276    13.911857     5.373925    19.677668   25.940901   
2020-01-08     4.109737    13.514375     5.509529    19.978729   25.875559   

            600048_保利发展  000858_五粮液  600519_贵州茅台  601857_中国石油  600028_中国石化  
date                                                                        
2020-01-02    12.602703  111.862382   979.045560     4.377881     3.524560  
2020-01-03    12.362430  110.566581   934.477327     4.437546     3.592733  
2020-01-06    12.153160  109.423227   933.983472     4.646371     3.654089  
2020-01-07    12.238418  109.5

In [9]:
# 用 pd.melt 转换为长表（date, code, close）
long_df = wide_df.reset_index().melt(
    id_vars='date',
    var_name='code_name',
    value_name='close'
)
long_df[['code', 'name']] = long_df['code_name'].str.split('_', n=1, expand=True)
long_df = long_df[['date', 'code', 'name', 'close']]
long_df = long_df.dropna(subset=['close'])

print('=== 长表（Long Format）===')
print(f'形状: {long_df.shape}')
print(long_df.head(10))
print(f'\n长表适合的操作：')
print('- 分组聚合（按股票代码计算统计量）')
print('- 与其他长格式数据（如宏观数据）合并')
print('- seaborn绑图（hue参数自动分组）')
print('- 面板数据回归分析')

=== 长表（Long Format）===
形状: (15440, 4)
        date    code  name     close
0 2020-01-02  601398  工商银行  4.151460
1 2020-01-03  601398  工商银行  4.165368
2 2020-01-06  601398  工商银行  4.151460
3 2020-01-07  601398  工商银行  4.179276
4 2020-01-08  601398  工商银行  4.109737
5 2020-01-09  601398  工商银行  4.109737
6 2020-01-10  601398  工商银行  4.109737
7 2020-01-13  601398  工商银行  4.123645
8 2020-01-14  601398  工商银行  4.130599
9 2020-01-15  601398  工商银行  4.088876

长表适合的操作：
- 分组聚合（按股票代码计算统计量）
- 与其他长格式数据（如宏观数据）合并
- seaborn绑图（hue参数自动分组）
- 面板数据回归分析


---
## 3.3 多表合并

In [10]:
# 读取指数数据
hs300 = pd.read_csv('data/index/index_000300.csv')
hs300['日期'] = pd.to_datetime(hs300['日期'])
hs300 = hs300.set_index('日期')
# 转换为数值型
for col in hs300.columns:
    hs300[col] = pd.to_numeric(hs300[col], errors='coerce')
hs300 = hs300.rename(columns={
    '开盘价': 'hs300_open', '收盘价': 'hs300_close',
    '最高价': 'hs300_high', '最低价': 'hs300_low',
    '成交量': 'hs300_volume', '成交额': 'hs300_amount'
})

zz500 = pd.read_csv('data/index/index_000905.csv')
zz500['日期'] = pd.to_datetime(zz500['日期'])
zz500 = zz500.set_index('日期')
for col in zz500.columns:
    zz500[col] = pd.to_numeric(zz500[col], errors='coerce')
zz500 = zz500.rename(columns={
    '开盘价': 'zz500_open', '收盘价': 'zz500_close',
    '最高价': 'zz500_high', '最低价': 'zz500_low',
    '成交量': 'zz500_volume', '成交额': 'zz500_amount'
})

print(f'沪深300: {hs300.shape[0]} 行')
print(f'中证500: {zz500.shape[0]} 行')

沪深300: 1544 行
中证500: 1544 行


In [11]:
# 合并1：个股日度数据与指数日度数据按日期做 left join
combined_stocks = []

for code in stock_dfs:
    df = stock_dfs[code].copy()
    rows_before = len(df)
    df['code'] = code
    df['name'] = stock_info[code][0]
    df['industry'] = stock_info[code][1]
    # 与沪深300合并
    df = df.join(hs300[['hs300_close', 'hs300_volume']], how='left')
    # 与中证500合并
    df = df.join(zz500[['zz500_close', 'zz500_volume']], how='left')
    rows_after = len(df)
    combined_stocks.append(df)
    print(f'{stock_info[code][0]}({code}): 合并前 {rows_before} 行, 合并后 {rows_after} 行 (left join，行数不变)')

print(f'\n共合并 {len(combined_stocks)} 只股票')

工商银行(601398): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
平安银行(000001): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
长安汽车(000625): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
上汽集团(600104): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
万科A(000002): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
保利发展(600048): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)


五粮液(000858): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
贵州茅台(600519): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
中国石油(601857): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)
中国石化(600028): 合并前 1544 行, 合并后 1544 行 (left join，行数不变)

共合并 10 只股票


In [12]:
# 合并2：月度宏观数据与日度数据合并
# 处理频率不一致：将月度宏观数据映射到对应月份的每个交易日

# 读取CPI数据
cpi = pd.read_csv('data/macro/macro_cpi.csv')
cpi['date'] = pd.to_datetime(cpi['date'])
cpi['year_month'] = cpi['date'].dt.to_period('M').astype(str)
cpi_map = dict(zip(cpi['year_month'], cpi['cpi_yoy']))

# 读取M2数据
m2 = pd.read_csv('data/macro/macro_m2.csv')
m2['date'] = pd.to_datetime(m2['date'])
m2['year_month'] = m2['date'].dt.to_period('M').astype(str)
m2_map = dict(zip(m2['year_month'], m2['m2_yoy']))

print(f'CPI月度数据: {len(cpi)} 条, 时间范围: {cpi["date"].min()} ~ {cpi["date"].max()}')
print(f'M2月度数据: {len(m2)} 条, 时间范围: {m2["date"].min()} ~ {m2["date"].max()}')

CPI月度数据: 70 条, 时间范围: 2020-01-09 00:00:00 ~ 2025-09-10 00:00:00
M2月度数据: 76 条, 时间范围: 2020-01-01 00:00:00 ~ 2026-04-01 00:00:00


In [13]:
# 将宏观数据映射到每个股票的日度数据
all_stock_combined_list = []

for i, df in enumerate(combined_stocks):
    df = df.copy()
    rows_before = len(df)
    # 映射CPI和M2：将日度日期转为年月，再映射到宏观数据
    df['year_month'] = df.index.to_period('M').astype(str)
    df['cpi_yoy'] = df['year_month'].map(cpi_map)
    df['m2_yoy'] = df['year_month'].map(m2_map)
    df = df.drop(columns=['year_month'])
    rows_after = len(df)
    all_stock_combined_list.append(df)
    
    cpi_cover = df['cpi_yoy'].notna().sum()
    m2_cover = df['m2_yoy'].notna().sum()
    if i == 0:
        print(f'{stock_info[codes[0]][0]}({codes[0]}): 行数 {rows_before} -> {rows_after}, CPI覆盖{cpi_cover}行, M2覆盖{m2_cover}行')

# 合并所有股票
all_stock_combined = pd.concat(all_stock_combined_list)
print(f'\n最终合并数据形状: {all_stock_combined.shape}')
print(f'行数变化说明：使用map映射方式将月度宏观数据填充到对应月份的每个交易日，行数不变')
print(f'CPI覆盖率: {all_stock_combined["cpi_yoy"].notna().sum()}/{len(all_stock_combined)} ({all_stock_combined["cpi_yoy"].notna().mean()*100:.1f}%)')
print(f'M2覆盖率: {all_stock_combined["m2_yoy"].notna().sum()}/{len(all_stock_combined)} ({all_stock_combined["m2_yoy"].notna().mean()*100:.1f}%)')

工商银行(601398): 行数 1544 -> 1544, CPI覆盖1355行, M2覆盖1532行



最终合并数据形状: (15440, 17)
行数变化说明：使用map映射方式将月度宏观数据填充到对应月份的每个交易日，行数不变
CPI覆盖率: 13550/15440 (87.8%)


M2覆盖率: 15320/15440 (99.2%)


---
## 方式A：CSV存储

将清洗后的数据保存为CSV格式。

In [14]:
# 保存清洗后的个股数据（CSV）
for code in stock_dfs:
    df = stock_dfs[code]
    filepath = f'data/clean/stock_{code}_clean.csv'
    df.to_csv(filepath, encoding='utf-8-sig')
    print(f'已保存: {filepath} ({df.shape[0]} 行)')

# 保存长格式清洗数据
long_df.to_csv('data/clean/stock_clean.csv', index=False, encoding='utf-8-sig')
print(f'已保存: data/clean/stock_clean.csv ({long_df.shape[0]} 行)')

# 保存合并数据
all_stock_combined.to_csv('data/combined/combined_data.csv', encoding='utf-8-sig')
print(f'已保存: data/combined/combined_data.csv ({all_stock_combined.shape[0]} 行)')

print('\nCSV存储完成！')

已保存: data/clean/stock_601398_clean.csv (1544 行)


已保存: data/clean/stock_000001_clean.csv (1544 行)


已保存: data/clean/stock_000625_clean.csv (1544 行)
已保存: data/clean/stock_600104_clean.csv (1544 行)


已保存: data/clean/stock_000002_clean.csv (1544 行)
已保存: data/clean/stock_600048_clean.csv (1544 行)
已保存: data/clean/stock_000858_clean.csv (1544 行)


已保存: data/clean/stock_600519_clean.csv (1544 行)
已保存: data/clean/stock_601857_clean.csv (1544 行)
已保存: data/clean/stock_600028_clean.csv (1544 行)


已保存: data/clean/stock_clean.csv (15440 行)


已保存: data/combined/combined_data.csv (15440 行)

CSV存储完成！


---
## 方式C：SQLite存储（进阶）

将清洗后的数据存入SQLite数据库，设计合理的表结构。

In [15]:
# 创建SQLite数据库
db_path = 'data/combined/fin_data.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)

# 创建表1：stock_price（股票行情）
conn.execute('''
    CREATE TABLE IF NOT EXISTS stock_price (
        code    TEXT,
        date    TEXT,
        open    REAL,
        close   REAL,
        high    REAL,
        low     REAL,
        volume  REAL,
        amount  REAL,
        log_return REAL,
        is_extreme INTEGER,
        hs300_close REAL,
        zz500_close REAL,
        cpi_yoy REAL,
        m2_yoy REAL,
        PRIMARY KEY (code, date)
    )
''')

# 创建表2：macro_data（宏观数据）
conn.execute('''
    CREATE TABLE IF NOT EXISTS macro_data (
        date      TEXT,
        indicator TEXT,
        value     REAL,
        PRIMARY KEY (date, indicator)
    )
''')

# 创建表3：stock_info（股票信息）
conn.execute('''
    CREATE TABLE IF NOT EXISTS stock_info (
        code     TEXT PRIMARY KEY,
        name     TEXT,
        industry TEXT
    )
''')

conn.commit()
print('数据库表结构创建完成！')
print('- stock_price: 股票行情数据（含指数和宏观数据）')
print('- macro_data: 宏观经济指标')
print('- stock_info: 股票基本信息')

数据库表结构创建完成！
- stock_price: 股票行情数据（含指数和宏观数据）
- macro_data: 宏观经济指标
- stock_info: 股票基本信息


In [16]:
# 插入股票行情数据 - 使用外部脚本创建数据库（避免编码问题）
# 先关闭当前连接
conn.close()

# 运行数据库创建脚本
%run create_db.py

# 重新连接
conn = sqlite3.connect(db_path)

Inserting 15440 rows into stock_price
stock_price inserted!


stock_info inserted!
macro_data inserted!

=== JOIN Test ===
         date    code     close  cpi
0  2020-01-02  601398  4.151460  4.5
1  2020-01-03  601398  4.165368  4.5
2  2020-01-06  601398  4.151460  4.5
3  2020-01-07  601398  4.179276  4.5
4  2020-01-08  601398  4.109737  4.5

Database created successfully!


In [17]:
# 插入宏观数据（已在create_db.py中完成，此处仅查询确认）
macro_count = pd.read_sql_query('SELECT COUNT(*) as cnt FROM macro_data', conn)
print(f'宏观数据已在数据库中: {macro_count["cnt"].values[0]} 条记录')

# 插入股票信息（已在create_db.py中完成，此处仅查询确认）
info_count = pd.read_sql_query('SELECT COUNT(*) as cnt FROM stock_info', conn)
print(f'股票信息已在数据库中: {info_count["cnt"].values[0]} 条记录')

conn.commit()

宏观数据已在数据库中: 144 条记录
股票信息已在数据库中: 10 条记录


### SQLite 查询演示

In [18]:
# 跨表JOIN演示：将股票行情与宏观数据按月份合并
query = """
SELECT p.date, p.code, p.close,
       m.value AS cpi
FROM stock_price p
LEFT JOIN macro_data m
       ON substr(p.date, 1, 7) = substr(m.date, 1, 7)
      AND m.indicator = 'cpi'
LIMIT 10
"""
df_join = pd.read_sql_query(query, conn)
print('=== 跨表JOIN演示 ===')
print(df_join)

=== 跨表JOIN演示 ===
         date    code     close  cpi
0  2020-01-02  601398  4.151460  4.5
1  2020-01-03  601398  4.165368  4.5
2  2020-01-06  601398  4.151460  4.5
3  2020-01-07  601398  4.179276  4.5
4  2020-01-08  601398  4.109737  4.5
5  2020-01-09  601398  4.109737  4.5
6  2020-01-10  601398  4.109737  4.5
7  2020-01-13  601398  4.123645  4.5
8  2020-01-14  601398  4.130599  4.5
9  2020-01-15  601398  4.088876  4.5


In [19]:
# 自定义SQL查询1：查询每个行业中年均收盘价最高的股票
query1 = """
SELECT s.industry, s.name, p.code,
       AVG(p.close) AS avg_close,
       AVG(p.volume) AS avg_volume
FROM stock_price p
JOIN stock_info s ON p.code = s.code
WHERE p.date >= '2023-01-01'
GROUP BY s.industry, p.code
ORDER BY s.industry, avg_close DESC
"""
df_q1 = pd.read_sql_query(query1, conn)
print('=== 查询1：各行业股票年均收盘价（2023年以来）===')
print('用途说明：比较同一行业内不同股票的价格水平，识别行业龙头')
print(df_q1.to_string(index=False))

=== 查询1：各行业股票年均收盘价（2023年以来）===
用途说明：比较同一行业内不同股票的价格水平，识别行业龙头
industry name   code   avg_close   avg_volume
     房地产 保利发展 600048    9.299795 1.183562e+08
      汽车 上汽集团 600104   14.832961 4.359515e+07
      白酒 贵州茅台 600519 1499.928741 3.168994e+06
      能源 中国石油 601857    7.997932 1.739300e+08
      能源 中国石化 600028    5.596825 1.666054e+08
      银行 工商银行 601398    5.572247 3.134501e+08


In [20]:
# 自定义SQL查询2：查询成交量排名前5的交易日及对应股票
query2 = """
SELECT p.date, s.name, s.industry, p.volume, p.close,
       p.cpi_yoy
FROM stock_price p
JOIN stock_info s ON p.code = s.code
ORDER BY p.volume DESC
LIMIT 10
"""
df_q2 = pd.read_sql_query(query2, conn)
print('=== 查询2：成交量最大的交易日及对应股票 ===')
print('用途说明：识别市场交易最活跃的时点，可能与重大事件或政策发布有关，')
print('同时观察这些时点的CPI水平，探索交易活跃度与宏观环境的关联')
print(df_q2.to_string(index=False))

conn.close()
print('\nSQLite数据库操作完成！')

=== 查询2：成交量最大的交易日及对应股票 ===
用途说明：识别市场交易最活跃的时点，可能与重大事件或政策发布有关，
同时观察这些时点的CPI水平，探索交易活跃度与宏观环境的关联
      date name industry       volume     close  cpi_yoy
2024-10-08 工商银行       银行 1688226172.0  5.639041      0.4
2024-09-30 工商银行       银行 1616174183.0  5.694326      0.6
2026-03-03 中国石化       能源 1502438208.0  7.820000      NaN
2020-11-30 工商银行       银行 1245663648.0  3.892218      0.5
2026-03-04 中国石化       能源 1233814563.0  7.400000      NaN
2024-10-10 工商银行       银行 1091528147.0  5.777253      0.4
2026-03-02 中国石化       能源 1088572035.0  7.110000      NaN
2026-03-09 中国石化       能源 1077922429.0  7.000000      NaN
2026-03-04 中国石油       能源 1074717349.0 13.240000      NaN
2023-05-08 工商银行       银行 1048861302.0  4.353521      0.1

SQLite数据库操作完成！


### 如何重建数据库

1. 运行 `01_download.ipynb` 下载原始数据
2. 运行本 Notebook 中所有单元格
3. `fin_data.db` 将自动生成在 `data/combined/` 目录下

数据库文件未上传至GitHub（已在.gitignore中排除），需通过运行Notebook从头重建。